# Fish Speech 1.5 Emotion Inference Only

이 노트북은 학습이 끝난 모델을 사용해 텍스트와 감정만으로 짧은 wav를 생성하는 추론 전용 Colab 노트북입니다.

학습, 전처리, VQ 추출, protobuf 생성, LoRA merge는 수행하지 않습니다.

## 0. Required Drive Artifacts

팀 Colab 계정의 Google Drive에 아래 폴더가 있어야 합니다.

```text
/content/drive/MyDrive/gyul-ai/emotion-tts/checkpoints/fish-speech-1.5
/content/drive/MyDrive/gyul-ai/emotion-tts/checkpoints/fish-speech-1.5-aihub-emotion-full
```

`fish-speech-1.5-aihub-emotion-full`은 fine-tuned text2semantic 모델이고, `fish-speech-1.5`는 tokenizer와 VQGAN decoder checkpoint를 제공합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/gyul-ai/emotion-tts')
CHECKPOINT_ROOT = DRIVE_ROOT / 'checkpoints'
BASE_CKPT = CHECKPOINT_ROOT / 'fish-speech-1.5'
MERGED_CKPT = CHECKPOINT_ROOT / 'fish-speech-1.5-aihub-emotion-full'
GENERATED_ROOT = DRIVE_ROOT / 'generated_samples' / 'team_inference'

REPO_DIR = Path('/content/Gyul-AI-Repository')
FISH_DIR = Path('/content/fish-speech-v15')

GENERATED_ROOT.mkdir(parents=True, exist_ok=True)

print('DRIVE_ROOT:', DRIVE_ROOT)
print('BASE_CKPT:', BASE_CKPT)
print('MERGED_CKPT:', MERGED_CKPT)
print('GENERATED_ROOT:', GENERATED_ROOT)

## 1. Clone Project Repo

추론 wrapper와 감정 태그 설정을 가져옵니다.

In [ ]:
REPO_URL = 'https://github.com/novvvv/Gyul-AI-Repository.git'
BRANCH = 'MaTuna/tts'

%cd /content
!rm -rf "{REPO_DIR}"
!git clone --branch "{BRANCH}" "{REPO_URL}" "{REPO_DIR}"
%cd /content/Gyul-AI-Repository
!git status --short --branch
!test -f scripts/run_fish15_emotion_inference.py
!test -f configs/emotion_tags.yaml

## 2. Install Fish Speech 1.5 Runtime

Colab 런타임은 세션마다 초기화되므로 Fish Speech 1.5 코드는 매번 다시 clone/install합니다.

In [ ]:
FISH_REPO = 'https://github.com/fishaudio/fish-speech.git'
FISH_TAG = 'v1.5.1'

%cd /content
!apt-get update -y
!apt-get install -y ffmpeg libsox-dev libsndfile1
!rm -rf "{FISH_DIR}"
!git clone --branch "{FISH_TAG}" --depth 1 "{FISH_REPO}" "{FISH_DIR}"
%cd /content/fish-speech-v15

import os, sys
os.environ['PYTHONPATH'] = f"{FISH_DIR}:{os.environ.get('PYTHONPATH', '')}"
if str(FISH_DIR) not in sys.path:
    sys.path.insert(0, str(FISH_DIR))

!python -m pip install --upgrade pip
!python -m pip install -e . --no-deps
!python -m pip install -U \
  'numpy<=1.26.4' 'transformers>=4.45.2,<4.58' 'datasets==2.18.0' \
  'lightning>=2.1.0' 'hydra-core>=1.3.2' 'tensorboard>=2.14.1' \
  'natsort>=8.4.0' 'einops>=0.7.0' 'librosa>=0.10.1' \
  'rich>=13.5.3,<14' 'grpcio>=1.58.0' 'loguru>=0.6.0' \
  'loralib>=0.1.2' 'pyrootutils>=1.0.4' 'vector_quantize_pytorch==1.14.24' \
  'resampy>=0.4.3' 'einx[torch]==0.2.2' 'zstandard>=0.22.0' \
  pydub 'opencc-python-reimplemented==0.1.7' ormsgpack 'tiktoken>=0.8.0' \
  'pydantic==2.9.2' cachetools soundfile
!python -m pip install -U 'protobuf>=4.25.1,<5'
!python -m pip uninstall -y torchvision

import torch, torchaudio, fish_speech
print('fish_speech:', fish_speech.__file__)
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
print('gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
!git log -1 --oneline

## 3. Verify Model Artifacts

모든 값이 `True`여야 학습 없이 바로 추론할 수 있습니다.

In [ ]:
required = [
    BASE_CKPT / 'model.pth',
    BASE_CKPT / 'config.json',
    BASE_CKPT / 'special_tokens.json',
    BASE_CKPT / 'tokenizer.tiktoken',
    BASE_CKPT / 'firefly-gan-vq-fsq-8x1024-21hz-generator.pth',
    MERGED_CKPT / 'model.pth',
    MERGED_CKPT / 'config.json',
    MERGED_CKPT / 'special_tokens.json',
    MERGED_CKPT / 'tokenizer.tiktoken',
]

missing = []
for path in required:
    ok = path.exists()
    print(ok, path)
    if not ok:
        missing.append(path)

assert not missing, 'Missing required artifacts: ' + ', '.join(map(str, missing))

## 4. Generate Wav

`EMOTION`과 `TEXT`만 바꿔 실행합니다.

지원 감정: `neutral`, `happy`, `sad`, `angry`, `anxious`, `hurt`, `embarrassed`

In [ ]:
from IPython.display import Audio, display
from pathlib import Path

# Change these two values.
EMOTION = 'happy'
TEXT = '오늘 정말 잘했어. 조금만 더 힘내보자.'

OUT_WAV = GENERATED_ROOT / f'{EMOTION}_sample.wav'

%cd /content/Gyul-AI-Repository
!python scripts/run_fish15_emotion_inference.py \
  --fish-speech-dir "{FISH_DIR}" \
  --checkpoint-dir "{MERGED_CKPT}" \
  --base-checkpoint-dir "{BASE_CKPT}" \
  --emotion "{EMOTION}" \
  --text "{TEXT}" \
  --output "{OUT_WAV}" \
  --half

print('saved:', OUT_WAV)
display(Audio(str(OUT_WAV)))